In [ ]:
# Imports
import tensorflow as tf;
import numpy as np;
import pandas;
import string;
import nltk;
import csv;
import sklearn;

from tensorflow.python.keras.activations import relu, sigmoid, softmax;
from tensorflow.keras.preprocessing.sequence import pad_sequences;
from tensorflow.keras.preprocessing.text import Tokenizer;
from tensorflow.keras.utils import to_categorical;
from tensorflow import keras;

nltk.download("punkt");

In [ ]:
# Processing data
data = pandas.read_csv("../input/all-trumps-twitter-insults-20152021/trump_insult_tweets_2014_to_2021.csv")["tweet"][0:5000];
data = [nltk.word_tokenize(i.lower().strip("“").rstrip("”")) for i in data];
maxlen = max([len(i) for i in data]);

In [ ]:
# Creating tokenizer
tokenizer = Tokenizer(num_words=100000, oov_token="<>");
tokenizer.fit_on_texts(data);

In [ ]:
# Creating training data
data_formatted = tokenizer.texts_to_sequences(data);
iosize = len(tokenizer.word_index) + 1;
training_x = [];
training_y = [];

for i in data_formatted:
  for v in range(1, len(i)):
    training_x.append(i[:v]);
    training_y.append(i[v]);

training_x = pad_sequences(training_x, maxlen=maxlen);
training_y = to_categorical(training_y, num_classes=len(tokenizer.word_index) + 1);

In [ ]:
# Shuffling training data
training_x, training_y = sklearn.utils.shuffle(training_x, training_y); 

In [ ]:
with tf.device("/device:GPU:0"):
  # Creating model
  model = keras.Sequential([
      keras.layers.Embedding(iosize, 64, input_length=maxlen),
      keras.layers.Bidirectional(keras.layers.LSTM(100)),
      keras.layers.Dense(iosize, activation=softmax),
  ]);     

  # Compiling model
  model.compile(loss="categorical_crossentropy", optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), metrics=["accuracy"]);

# Fitting model
model.fit(training_x, training_y, epochs=100, batch_size=32);
model.save("model.h5");

In [ ]:
seed = "a hero is".split();

for x in range(30):
    raw = model.predict(pad_sequences(tokenizer.texts_to_sequences([seed]), maxlen=maxlen))
    predict = np.argmax(raw[0]);

    for i in tokenizer.word_index:
        if (tokenizer.word_index[i] == predict):
            predict = i;
            break;

    seed.append(predict);

print(seed)